In [5]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os
import matplotlib.pylab as plt
from sklearn.metrics import roc_curve, roc_auc_score, recall_score

In [6]:
train_dir = r'/home/kchen/Documents/iom/data/split/train'
val_dir = r'/home/kchen/Documents/iom/data/split/val'
data_dir = r'/home/kchen/Documents/iom/data/jpg'
test_dir = r'/home/kchen/Documents/iom/data/split/test'

In [7]:
batch_size = 1
img_height = 512
img_width = 512

In [8]:
#load images into datasets with 'train' set split into training and validation
train_ds = tf.keras.preprocessing.image_dataset_from_directory(train_dir, label_mode='binary', subset='training', validation_split = 0.15, seed=0, image_size=(img_height, img_width), batch_size=batch_size, color_mode='rgb')
val_ds = tf.keras.preprocessing.image_dataset_from_directory(train_dir, label_mode='binary', subset='validation', validation_split = 0.15, seed=0, image_size=(img_height, img_width), batch_size=batch_size, color_mode='rgb')


Found 359 files belonging to 2 classes.
Using 306 files for training.
Found 359 files belonging to 2 classes.
Using 53 files for validation.


In [9]:
#normalize and apply data augmentation
preprocessing_model = tf.keras.Sequential()
do_data_augmentation = True
if do_data_augmentation:
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomRotation(40))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomTranslation(0.2, 0.2))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomZoom(0.2, 0.2))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomFlip(mode="horizontal"))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomFlip(mode="vertical"))


In [10]:
train_ds = train_ds.map(lambda images, labels:
                        (preprocessing_model(images), labels))


#only apply normalization to validation set
val_ds = val_ds.unbatch().batch(batch_size)
val_ds = val_ds.map(lambda images, labels:
                    (normalization_layer(images), labels))

In [11]:
#download Xception base model
base_model = keras.applications.xception.Xception(weights='imagenet', input_shape=(img_height, img_height, 3), include_top=False)
base_model.trainable = False

In [13]:
#add input layer and outputs, last layer is binary classifier
inputs = keras.Input(shape=(img_height, img_height, 3))
x = tf.keras.applications.xception.preprocess_input(inputs)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.4)(x)
outputs = keras.layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

In [14]:
#compile and train
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=[keras.metrics.BinaryAccuracy(), keras.metrics.AUC(), keras.metrics.Recall(), keras.metrics.TrueNegatives(), keras.metrics.FalseNegatives(), keras.metrics.TruePositives(), keras.metrics.FalsePositives()])

early_stopping = keras.callbacks.EarlyStopping(
    patience=3,
    min_delta=0.001,
    restore_best_weights=True,)

model.fit(train_ds, epochs=20, validation_data=val_ds, callbacks=early_stopping)


Epoch 1/20
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
306/306 [==============================] - ETA: 0s - loss: 0.7116 - binary_accuracy: 0.5000 - auc: 0.5005 - recall: 0.2979 - true_negatives: 111.0000 - false_negatives: 99.0000 - true_positives: 42.0000 - false_positives: 54.0000WARNING:tensorflow:AutoGraph could not transform <function Model.make_test

In [ ]:
#unfreeze and train at low learning rate
base_model.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(1e-5), 
    loss=keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[keras.metrics.BinaryAccuracy(), keras.metrics.AUC(), keras.metrics.Recall(), keras.metrics.TrueNegatives(), keras.metrics.FalseNegatives(), keras.metrics.TruePositives(), keras.metrics.FalsePositives()],
)

epochs = 10
model.fit(train_ds, epochs=epochs, validation_data=val_ds, callbacks=early_stopping)

Epoch 1/10
306/306 [==============================] - 77s 225ms/step - loss: 0.6567 - binary_accuracy: 0.6013 - auc_1: 0.6347 - recall_1: 0.4894 - true_negatives_1: 115.0000 - false_negatives_1: 72.0000 - true_positives_1: 69.0000 - false_positives_1: 50.0000 - val_loss: 0.6807 - val_binary_accuracy: 0.5283 - val_auc_1: 0.5986 - val_recall_1: 0.6000 - val_true_negatives_1: 13.0000 - val_false_negatives_1: 10.0000 - val_true_positives_1: 15.0000 - val_false_positives_1: 15.0000
Epoch 2/10
306/306 [==============================] - 66s 217ms/step - loss: 0.6575 - binary_accuracy: 0.6176 - auc_1: 0.6459 - recall_1: 0.5177 - true_negatives_1: 116.0000 - false_negatives_1: 68.0000 - true_positives_1: 73.0000 - false_positives_1: 49.0000 - val_loss: 0.6700 - val_binary_accuracy: 0.5849 - val_auc_1: 0.6193 - val_recall_1: 0.7600 - val_true_negatives_1: 12.0000 - val_false_negatives_1: 6.0000 - val_true_positives_1: 19.0000 - val_false_positives_1: 16.0000
Epoch 3/10
306/306 [=================

In [15]:
model.save('xcept.h5')

/home/kchen/.local/lib/python3.9/site-packages/tensorflow/python/keras/utils/generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '


In [16]:
model = tf.keras.models.load_model('xcept.h5')

In [17]:
#load test set and normalize using same code
test_ds = tf.keras.preprocessing.image_dataset_from_directory(test_dir, label_mode='binary', seed=0, image_size=(img_height, img_width), batch_size=batch_size, color_mode='rgb')

Found 91 files belonging to 2 classes.


In [18]:
#evaluate on test set using evaluate
model.evaluate(test_ds)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
91/91 [==============================] - 14s 142ms/step - loss: 0.6299 - binary_accuracy: 0.5934 - auc: 0.7116 - recall: 0.8333 - true_negatives: 19.0000 - false_negatives: 7.0000 - true_positives: 35.0000 - false_positives: 30.0000


[0.6298875212669373,
 0.593406617641449,
 0.7116131782531738,
 0.8333333134651184,
 19.0,
 7.0,
 35.0,
 30.0]

In [19]:
#generate predictions on test set
preds = model.predict(test_ds)
preds.shape

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


(91, 1)

In [20]:
preds = preds.flatten()
preds.shape

(91,)

In [21]:
preds

array([0.35679144, 0.41450825, 0.45510262, 0.51970625, 0.57716054,
       0.35785154, 0.75985026, 0.5036755 , 0.19824788, 0.5110021 ,
       0.63411224, 0.6378002 , 0.5491746 , 0.68465394, 0.33104062,
       0.58614945, 0.579817  , 0.59657437, 0.6147461 , 0.516993  ,
       0.6695434 , 0.54170424, 0.76524353, 0.73815066, 0.59906536,
       0.6780088 , 0.6482179 , 0.6891631 , 0.5139216 , 0.37898278,
       0.60062355, 0.5507093 , 0.50547516, 0.63221204, 0.32634175,
       0.7909235 , 0.53482175, 0.50991344, 0.5208272 , 0.49451584,
       0.5990774 , 0.1070106 , 0.56294435, 0.7305989 , 0.40151185,
       0.59511566, 0.6719544 , 0.5360674 , 0.20194653, 0.5663659 ,
       0.2667804 , 0.546893  , 0.7171006 , 0.71201664, 0.67995644,
       0.6581387 , 0.58763385, 0.49715862, 0.3228504 , 0.2274329 ,
       0.395464  , 0.5819842 , 0.58378685, 0.6868837 , 0.5190313 ,
       0.4541776 , 0.70066273, 0.608709  , 0.478694  , 0.57043564,
       0.61520004, 0.19252634, 0.6689503 , 0.48093086, 0.45223

In [22]:
#get labels by iterating through test set and appending labels to empty array
test_labels = np.array([])
for images, labels in test_ds:
    test_labels = np.append(test_labels, labels[0].numpy(), axis=0)
test_labels.shape

(91,)

In [23]:
test_labels

array([0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1.,
       1., 0., 0., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 0., 1., 1., 1.,
       0., 1., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 1., 1., 0., 1., 0., 0., 0., 1., 1., 1., 1., 1., 1., 0., 0.,
       1., 0., 1., 0., 1., 1., 0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 1.,
       1., 0., 1., 1., 0., 0.])

In [24]:
#seems like the labels and preds don't match up
roc_auc_score(test_labels, preds)

0.6175898931000972